# VigiLens on Google Colab (T4) - VidLLVIP Dataset

This notebook trains and evaluates VigiLens on the VidLLVIP paired RGB-IR video dataset:

1. Check the GPU and enable FP16 AMP.
2. Clone the repository and install dependencies.
3. Download and prepare VidLLVIP dataset.
4. Precompute RGB/IR features once.
5. Train the fusion + causal LSTM on cached features.
6. Calibrate the anomaly threshold on normal validation data.
7. Qualitative anomaly-score analysis (no ground-truth labels in VidLLVIP).

**Runtime:** select `Runtime -> Change runtime type -> T4 GPU` before running.

**Note:** VidLLVIP does not provide frame-level anomaly labels. This notebook trains on normal-only data and provides qualitative analysis. For quantitative evaluation (AUC/AP/FPR), use a labeled dataset like ShanghaiTech or Avenue.

In [ ]:
import os, sys, json, shutil
from pathlib import Path

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))
    torch.backends.cudnn.benchmark = True
else:
    print("No GPU detected. Cached training still works on CPU, but it will be slower.")

In [ ]:
# Change this if you forked the repository.
REPO_URL = "https://github.com/soumyaranjan4446/VigiLens.git"
PROJECT_DIR = "/content/VigiLens"

if not Path(PROJECT_DIR).exists():
    !git clone {REPO_URL} {PROJECT_DIR}

%cd {PROJECT_DIR}
!git rev-parse --short HEAD

In [ ]:
# Colab already provides a CUDA build of torch/torchvision; pip keeps it.
!pip install -q -r requirements.txt

import sklearn, cv2, ultralytics, streamlit
print("opencv", cv2.__version__)
print("ultralytics", ultralytics.__version__)
print("streamlit", streamlit.__version__)

In [ ]:
# Optional: persist checkpoints and results in Google Drive.
USE_DRIVE = False
DRIVE_DIR = None

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/VigiLens_runs")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    print("Drive export directory:", DRIVE_DIR)
else:
    print("Drive export disabled. Set USE_DRIVE=True to enable it.")

## VidLLVIP Dataset Preparation

VidLLVIP is a paired RGB-IR video dataset. Download and extract it, then prepare for VigiLens.

**Dataset structure:**
```
VidLLVIP/
    vi/      (visible videos)
    ir/      (infrared videos)
```

Files with identical names in `vi/` and `ir/` are corresponding pairs.

### Step 1: Download VidLLVIP

Download VidLLVIP from: https://github.com/jianfeng0369/VidLLVIP

Extract to `/content/VidLLVIP` (or upload manually).

### Step 2: Prepare dataset for VigiLens

This script:
- Discovers RGB/IR pairs by exact filename matching
- Validates video readability
- Creates source-level train/val/test splits to prevent data leakage
- Converts to MP4 format under `Data/`

In [ ]:
# Prepare VidLLVIP dataset
# Adjust --val-ratio and --test-ratio based on your needs
# Use --max-frames for smoke testing (small subset)

VIDLLVIP_PATH = "/content/VidLLVIP"  # Adjust if uploaded elsewhere

!python scripts/prepare_vidllvip.py \
    --input {VIDLLVIP_PATH} \
    --output Data \
    --val-ratio 0.2 \
    --test-ratio 0.1 \
    --seed 42

## Dataset Validation

Verify the prepared dataset structure and splits.

In [ ]:
import json
from pathlib import Path

# Load the VidLLVIP manifest
manifest_path = Path("Data/vidllvip_pairs.json")
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    
    print(f"Dataset: {manifest['dataset']}")
    print(f"Split method: {manifest['split_method']}")
    print(f"Val ratio: {manifest['val_ratio']}")
    print(f"Test ratio: {manifest['test_ratio']}")
    print(f"Seed: {manifest['seed']}")
    print()
    
    # Count pairs per split
    from collections import Counter
    splits = Counter(p['split'] for p in manifest['pairs'])
    print("Pairs per split:")
    for split, count in splits.items():
        print(f"  {split}: {count}")
    
    # Check source distribution
    sources = {}
    for p in manifest['pairs']:
        source = p['source_id']
        if source not in sources:
            sources[source] = []
        sources[source].append(p['split'])
    
    print(f"\nUnique sources: {len(sources)}")
    print("Source distribution:")
    from collections import defaultdict
    source_splits = defaultdict(Counter)
    for source, splits_list in sources.items():
        for split in splits_list:
            source_splits[source][split] += 1
    
    # Show a few examples
    for i, (source, counter) in enumerate(list(source_splits.items())[:5]):
        print(f"  {source}: {dict(counter)}")
    if len(source_splits) > 5:
        print(f"  ... and {len(source_splits) - 5} more")
else:
    print("Manifest not found. Run the preparation script first.")

## Smoke Test (Real VidLLVIP Data)

Run a quick smoke test with a small subset of VidLLVIP data to verify the pipeline works before full training.

In [ ]:
# Smoke test: prepare a small subset and precompute features
# This uses --max-frames to limit processing time

!rm -rf Data_smoke cache_smoke
!python scripts/prepare_vidllvip.py \
    --input {VIDLLVIP_PATH} \
    --output Data_smoke \
    --val-ratio 0.2 \
    --test-ratio 0.1 \
    --max-frames 50 \
    --all-train

In [ ]:
# Precompute features for smoke test (no segmentation for speed)
!python -m training.precompute \
    --data-dir Data_smoke \
    --output cache_smoke \
    --device {DEVICE} \
    --batch-size 16 \
    --no-segmentation

In [ ]:
# Quick training on smoke test data
!python -m training.train \
    --cache-dir cache_smoke \
    --epochs 2 \
    --batch-size 4 \
    --seq-len 16 \
    --device {DEVICE} \
    --checkpoint-dir checkpoints_smoke

In [ ]:
print("Smoke test completed successfully!")
print("If the above cells ran without errors, the pipeline is working.")
print("Now proceed to full training with the complete VidLLVIP dataset.")

In [ ]:
## Full VidLLVIP Training Pipeline

### Step 1: Precompute Features

Extract RGB/IR features for all VidLLVIP clips. This is done once and cached.

In [ ]:
# Precompute features for full VidLLVIP dataset
# Remove --max-frames for full dataset
# Use --no-segmentation only if YOLO is not needed or for faster testing

!rm -rf cache
!python -m training.precompute \
    --data-dir Data \
    --output cache \
    --device {DEVICE} \
    --batch-size 16 \
    --splits train,val

In [ ]:
## Step 2: Train Fusion + LSTM

Train the modality-aware fusion and causal LSTM on the cached features.

# Train on VidLLVIP cached features
# Adjust epochs based on your needs
# The ResNet encoders are frozen; only fusion + LSTM are trained

!python -m training.train \
    --cache-dir cache \
    --epochs 30 \
    --batch-size 8 \
    --seq-len 16 \
    --device {DEVICE} \
    --checkpoint-dir checkpoints

## Step 3: Calibrate Anomaly Threshold

Calibrate the threshold using normal validation data. Since VidLLVIP does not provide anomaly labels, we assume all validation clips are normal for threshold calibration purposes.

In [ ]:
# Calibrate threshold on validation data
# Note: VidLLVIP does not provide anomaly labels, so we assume validation clips are normal
# This threshold will be used for qualitative anomaly scoring

!python -m training.calibration \
    --cache-dir cache/val \
    --checkpoint checkpoints/best_model.pth \
    --output checkpoints/calibration.json \
    --device {DEVICE} \
    --k-std 3.0 \
    --percentile 95

In [ ]:
## Step 4: Qualitative Analysis (No Ground Truth)

Since VidLLVIP does not provide anomaly labels, we perform qualitative analysis on the test set.

In [ ]:
# Precompute test features for qualitative analysis
!python -m training.precompute \
    --data-dir Data \
    --output cache/test \
    --device {DEVICE} \
    --batch-size 16 \
    --splits test

# Run qualitative scoring on test set
# This will output anomaly scores but no quantitative metrics (no ground truth)

import torch
from models.anomaly_model import VigiLensModel
from training.scoring import prediction_errors
from pathlib import Path
from tqdm import tqdm
import json

# Load model
model = VigiLensModel(device=DEVICE, load_extractors=False)
model.load_checkpoint("checkpoints/best_model.pth")
model.to(DEVICE)
model.eval()

# Load calibration
with open("checkpoints/calibration.json") as f:
    calibration_info = json.load(f)
threshold = calibration_info["threshold"]

# Discover test caches
test_caches = sorted(Path("cache/test").rglob("*.pt"))
print(f"Found {len(test_caches)} test cache files")

# Collect anomaly scores
all_scores = []
for cache_path in tqdm(test_caches, desc="Scoring test"):
    cache = torch.load(cache_path, map_location=DEVICE)
    rgb_features = cache["rgb_features"].to(DEVICE)
    ir_features = cache["ir_features"].to(DEVICE) if cache["has_ir"] else None
    sequence_id = cache["sequence_id"]
    
    # Run through model
    model.reset_stream()
    scores = []
    with torch.no_grad():
        for i in range(rgb_features.shape[0]):
            rgb_feat = rgb_features[i:i+1]
            ir_feat = ir_features[i:i+1] if ir_features is not None else None
            
            fused = model.fuse_features(rgb_feat, ir_feat, True, ir_features is not None)
            prediction, (h, c) = model.forward(fused, model.hidden_state)
            model.hidden_state = (h.detach(), c.detach())
            
            if model.previous_prediction is not None:
                error = torch.nn.functional.mse_loss(model.previous_prediction, fused).item()
                scores.append(error)
            model.previous_prediction = prediction.detach()
    
    all_scores.extend(scores)
    
    # Basic stats per clip
    if scores:
        print(f"{sequence_id}: {len(scores)} frames, mean_score={sum(scores)/len(scores):.4f}, max_score={max(scores):.4f}")

print(f"\nOverall: {len(all_scores)} frames scored")
print(f"Calibrated threshold: {threshold:.4f}")
print(f"Frames above threshold: {sum(1 for s in all_scores if s > threshold)}/{len(all_scores)}")

In [ ]:
## Optional: Save to Google Drive

If you want to persist checkpoints and results to Google Drive.

if USE_DRIVE:
    import shutil
    for name in ["checkpoints", "cache"]:
        src = Path(name)
        dst = DRIVE_DIR / name
        if src.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f"Copied {src} -> {dst}")

In [ ]:
!python -m training.calibration \
    --cache-dir cache/val \
    --checkpoint checkpoints/best_model.pth \
    --output checkpoints/calibration.json \
    --device {DEVICE} \
    --k-std 3.0 \
    --percentile 95

## Evaluate with frame-level metrics (synthetic fallback)

In [ ]:
!python -m training.evaluate \
    --manifest Evaluation/manifest.json \
    --cache-dir cache/eval \
    --checkpoint checkpoints/best_model.pth \
    --calibration checkpoints/calibration.json \
    --output-dir Evaluation/results \
    --device {DEVICE} \
    --smoothing-window 5

In [ ]:
import pandas as pd

results = json.loads(Path("Evaluation/results/results.json").read_text(encoding="utf-8"))
summary = {
    "videos": results["num_videos"],
    "frames": results["num_frames"],
    "threshold": results["threshold"],
    "auc": results["metrics"]["auc"],
    "ap": results["metrics"]["ap"],
    "precision": results["metrics"]["precision"],
    "recall": results["metrics"]["recall"],
    "f1": results["metrics"]["f1"],
    "fpr": results["metrics"]["fpr"],
    "fpr_at_95tpr": results["metrics"]["fpr_at_95tpr"],
}
print(json.dumps(summary, indent=2))

scores = pd.read_csv("Evaluation/results/scores.csv")
display(scores.head())
display(scores.groupby("sequence_id")["smoothed_score"].agg(["count", "mean", "max"]))

## Run the Streamlit app (optional)

Colab cannot expose port 8501 directly. Use a tunnel such as `cloudflared` or `ngrok` if you want the UI:

```bash
!pip install -q streamlit cloudflared
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 &
# then start a tunnel to localhost:8501 in another cell
```

The app needs `checkpoints/best_model.pth`, `checkpoints/calibration.json`, and `yolov8n-seg.pt` (downloaded automatically on first use).

## Next steps

- Replace the synthetic evaluation set with labeled anomaly clips.
- Recalibrate whenever the camera/domain changes.
- Train longer on KAIST normal data, then fine-tune stage 4 with `--online --unfreeze-layer4` if GPU memory allows.
- Compare `metrics_raw` and `metrics_smoothed`; smoothing trades latency for stability.